# Bitcoin: data collection

Section 3.1 of the paper. Three label sources are combined:

* **ChainAbuse**: community scam reports for Bitcoin addresses (fraud).
* **Elliptic++**: the wallet labels released with the Elliptic++ dataset (`class 1` illicit, `class 2` licit).
* **Recent sample**: addresses observed in the unconfirmed-transaction pool of blockchain.info in early April 2025 (licit reference).

For every address the full transaction history is downloaded from the blockchain.info `rawaddr` endpoint and turned into 21 wallet features (`src/bfx/bitcoin.py`).

The download cells need network access and take days for the full set, so they are switched off by default. None of the raw data is redistributed here; `data/README.md` lists where each file comes from.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(ROOT / "src"))

import re

import pandas as pd

from bfx import bitcoin

DATA = ROOT / "data" / "bitcoin"
DOWNLOAD = False

## Labels

ChainAbuse reports are free text; we keep the strings that are syntactically valid Bitcoin addresses (legacy, P2SH or bech32) and drop repeated addresses. The export is expected at `data/bitcoin/labels/chainabuse_reports.csv` with an `address` column.

In [ ]:
BTC_ADDRESS = re.compile(r"^(1[a-km-zA-HJ-NP-Z1-9]{25,34}|3[a-km-zA-HJ-NP-Z1-9]{25,34}|bc1[ac-hj-np-z02-9]{11,71})$")


def clean_reports(path):
    reports = pd.read_csv(path)
    addresses = reports["address"].astype(str).str.strip()
    return addresses[addresses.str.match(BTC_ADDRESS)].drop_duplicates().tolist()


labels = DATA / "labels"
if (labels / "chainabuse_reports.csv").exists():
    chainabuse = clean_reports(labels / "chainabuse_reports.csv")
    print(f"ChainAbuse addresses: {len(chainabuse):,}")

Elliptic++ ships `wallets_classes.csv` ([git-disl/EllipticPlusPlus](https://github.com/git-disl/EllipticPlusPlus), folder `Actors Dataset`). All 14,266 illicit wallets were queried. Licit wallets were queried in file order and the download was stopped after 7,377 of them, which is the licit set used in the paper.

In [ ]:
ELLIPTIC_LICIT_LIMIT = 7_377

if (labels / "wallets_classes.csv").exists():
    elliptic = pd.read_csv(labels / "wallets_classes.csv")
    elliptic_fraud = elliptic.loc[elliptic["class"] == 1, "address"].tolist()
    elliptic_licit = elliptic.loc[elliptic["class"] == 2, "address"].tolist()[:ELLIPTIC_LICIT_LIMIT]
    print(f"Elliptic++ illicit: {len(elliptic_fraud):,}  licit used: {len(elliptic_licit):,}")

The recent sample is built by polling the unconfirmed-transaction pool every five seconds and collecting output addresses until 10,000 distinct ones are seen.

In [ ]:
if DOWNLOAD:
    recent = bitcoin.sample_recent_addresses(10_000)
    pd.Series(recent, name="address").to_csv(labels / "recent_sample.csv", index=False)

## Transaction histories

One JSON file per address, `<address>_transactions.json`, under `data/bitcoin/transactions/<source>/`. Requests that fail with a rate-limit or server error are retried with exponential back-off; other failures are skipped.

In [ ]:
if DOWNLOAD:
    import requests

    queues = {"chainabuse": chainabuse, "elliptic_fraud": elliptic_fraud,
              "elliptic_licit": elliptic_licit,
              "unknown": pd.read_csv(labels / "recent_sample.csv")["address"].tolist()}
    with requests.Session() as session:
        for source, addresses in queues.items():
            for address in addresses:
                bitcoin.fetch_wallet(address, DATA / "transactions" / source, session=session)

## Wallet features

Each transaction is expanded into (input address, output address) pairs carrying the output amount and the block time. Statistics are first computed per address appearing in the wallet's transactions and then aggregated:

| Group | Features |
|---|---|
| Structural | `total_in_degree`, `total_out_degree`, `total_unique_in`, `total_unique_out`, `in_out_ratio`, `unique_in_ratio`, `unique_out_ratio` |
| Monetary | `avg_in_transaction`, `avg_out_transaction`, `total_received`, `total_sent`, `net_balance`, `volume_ratio`, `net_balance_ratio`, `weighted_avg_tx` |
| Temporal | `avg_in_time_interval`, `avg_out_time_interval`, `avg_active_duration`, `activity_index`, `time_interval_ratio`, `wallet_lifetime_sec` |

Degrees count address pairs, not transactions; time intervals are in days. A per-address clustering coefficient is computed by the original code but not aggregated, so the model sees the 21 features above.

When transaction files are present in `data/bitcoin/transactions/`, the cell below recomputes their features and compares them with the stored feature tables.

In [ ]:
reference = {
    "chainabuse": "chainabuse_metrics.csv",
    "elliptic_fraud": "elliptic_fraud_metrics.csv",
    "elliptic_licit": "elliptic_licit_metrics.csv",
    "unknown": "recent_sample_metrics.csv",
}
checks = []
for source, table in reference.items():
    if not any((DATA / "transactions" / source).glob("*.json")):
        continue
    computed = bitcoin.features_from_json_dir(DATA / "transactions" / source).set_index("wallet")
    stored = pd.read_csv(DATA / "features" / table).set_index("wallet").loc[computed.index]
    gap = (computed[bitcoin.FEATURES] - stored[bitcoin.FEATURES]).abs().max().max()
    checks.append({"source": source, "wallets": len(computed), "max_abs_difference": gap})
pd.DataFrame(checks)

On the full download the same function is applied to every folder and the result written to `data/bitcoin/features/<source>_metrics.csv`, the input of the next notebook.

In [ ]:
if DOWNLOAD:
    for source, table in reference.items():
        bitcoin.features_from_json_dir(DATA / "transactions" / source).to_csv(DATA / "features" / table, index=False)